In [1]:
from google.colab import files
uploaded = files.upload()

Saving archive.zip to archive.zip


In [2]:
import zipfile
with zipfile.ZipFile('archive.zip', 'r') as z:
    z.extractall()
    print(z.namelist())

['tmdb_5000_credits.csv', 'tmdb_5000_movies.csv']


In [3]:
import pandas as pd
import numpy as np
import ast
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

df = movies.merge(credits.drop(columns=['title']), left_on='id', right_on='movie_id')
df = df[['title', 'overview', 'genres', 'keywords', 'tagline', 'cast', 'crew']]
df['overview'] = df['overview'].fillna('')
df['tagline'] = df['tagline'].fillna('')
print(df.shape)
df.head(2)

(4803, 7)


,title,overview,genres,keywords,tagline,cast,crew
0,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",Enter the World of Pandora.,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","At the end of the world, the adventure begins.","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [5]:
def get_names(text):
    return [i['name'] for i in ast.literal_eval(text)]

def get_top_cast(text):
    return [i['name'] for i in ast.literal_eval(text)][:3]

def get_director(text):
    return [i['name'] for i in ast.literal_eval(text) if i['job'] == 'Director']

df['genres'] = df['genres'].apply(get_names)
df['keywords'] = df['keywords'].apply(get_names)
df['cast'] = df['cast'].apply(get_top_cast)
df['crew'] = df['crew'].apply(get_director)

for col in ['genres', 'keywords', 'cast', 'crew']:
    df[col] = df[col].apply(lambda lst: [w.replace(" ", "") for w in lst])

df.head(2)

,title,overview,genres,keywords,tagline,cast,crew
0,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...",Enter the World of Pandora.,"[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron]
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","At the end of the world, the adventure begins.","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski]


In [6]:
df['tags'] = (df['overview'] + " " + df['tagline'] + " " +
              df['genres'].apply(" ".join) + " " +
              df['keywords'].apply(" ".join) + " " +
              df['cast'].apply(" ".join) + " " +
              df['crew'].apply(" ".join))

df['tags'] = df['tags'].str.lower()
df['tags'].iloc[0][:300]

'in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. enter the world of pandora. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetr'

In [7]:
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(df['tags']).toarray()

similarity = cosine_similarity(vectors)
print(similarity.shape)

(4803, 4803)


In [8]:
def recommend(movie_name):
    matches = df[df['title'].str.lower() == movie_name.lower()]
    if matches.empty:
        return ["Movie dataset mein nahi mili, dusra naam try karo"]
    idx = matches.index[0]
    scores = sorted(enumerate(similarity[idx]), key=lambda x: x[1], reverse=True)[1:6]
    return [df.iloc[i]['title'] for i, s in scores]

for m in recommend("The Avengers"):
    print(m)

Avengers: Age of Ultron
Captain America: Civil War
Iron Man 3
Captain America: The First Avenger
Ant-Man


In [9]:
!pip install gradio -q
import gradio as gr

def demo(movie_name):
    return "\n".join(f"{i+1}. {m}" for i, m in enumerate(recommend(movie_name)))

gr.Interface(fn=demo,
             inputs=gr.Textbox(label="Movie ka naam"),
             outputs=gr.Textbox(label="Top 5 Recommendations"),
             title="Movie Recommendation System").launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7aa57925cde7fbdab2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
